<u>#**Analysis of Polymarket weather markets**</u>
#Focusing on does the crowd favorite win?

First we need to obtain the data we want to work with for this analysis. To get the data there are three main API's that are free from polymarket, these are the GAMMA API, COLB API and DATA API. GAMMA API provides answers to questions like if a particular market exists on the site. COLB API provides info regarding the order books for markets and DATA API provides info about each user interacting with a market. 


In [ ]:
import requests
import pandas as pd
import numpy as np 
import json
import time 
from datetime import datetime, timezone, timedelta


def get_historic_events(days = 365):
    """This function pulls all event data regarding the events returning 2 data frames, the event df and the markets for a event """

    #create date range for scrapping
    end_date = datetime.now(timezone.utc)
    start_date = end_date - timedelta(days=days)

    offset = 0
    events = []

    while True:
    
        time.sleep(10)
        #filiters to get the info we want 
        params = {
            "tag_slug" : "weather", #only bring weather events
            "closed" : True, #event had ended 
            "limit" : 100, #requests a max of 100 events 
            "offset" : offset, #how far to move down
            "ascending" : False, #sort by newest to oldest
            "order" : end_date, #sort by the ending date 
            "end_date_min" : start_date.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "end_date_max" : end_date.strftime("%Y-%m-%dT%H:%M:%SZ") #create event range
            }

        events_response = requests.get(
            "https://gamma-api.polymarket.com/events",
            params=params)

        events_response.raise_for_status()

        #converts polymarket data into a dict
        batch = events_response.json()

        #extend not append to not get ladder of lists
        events.extend(batch)

        print(f"fetched: {len(events)} events")

        if len(batch) < 100:
            break
        else:
            offset += len(batch)


    #events are repeated markets ie some bet
    
    markets_rows = []
    events_clean = []

    for e in events:
        markets = e.pop("markets", []) or []
        events_clean.append(e)

        for m in markets:
            m["event_id"] = e.get("id")
            m["event_title"] = e.get("title")
            m["event_slug"] = e.get("slug")
            markets_rows.append(m)

    df_events = pd.DataFrame(events_clean)
    df_markets = pd.DataFrame(markets_rows)

    print(f"Total markets: {len(df_markets)}")

    return df_events, df_markets


df_events, df_markets = get_historic_events(days = 1)


fetched: 83 events
Total markets: 917


'32032084528449022253125230679116816503197270504605019685922635121350072489598'

In [21]:

#convert dates in 2026-09-05T12:00:00Z to unix format
def to_unix_timestamp(date_text):
    date = datetime.fromisoformat(
        date_text.replace("Z", "+00:00")
    )
    return int(date.timestamp())

In [ ]:
def price_history(clobtoken_id, start_ts, end_ts, fidelity):
    """This function gets all the market pricing history for a single market
    clobtoken_id is the clob token id 
    start_ts is is time range start for price history 
    end_ts is time range end for price history
    fidelity is the resolution we want to collect the data at
    """


    parms = {
        "market" : clobtoken_id,
        "startTs" : to_unix_timestamp(start_ts),
        "endTs": to_unix_timestamp(end_ts),
        "interval": "all",
        "fidelity": fidelity,
    }

    response = requests.get("https://clob.polymarket.com/prices-history", parms)
    response.raise_for_status()

    price_history = pd.DataFrame(response.json().get("history", []))

    if price_history.empty:
        return price_history
    else:
        price_history["timestamp"] = pd.to_datetime(price_history["t"], unit="s", utc=True)
        price_history["price"] = price_history["p"]
        return price_history[["timestamp", "price"]].sort_values("timestamp")



,timestamp,price
0,2026-09-03 05:00:17+00:00,0.2650
1,2026-09-03 06:00:19+00:00,0.1200
2,2026-09-03 07:00:17+00:00,0.1200
3,2026-09-03 08:00:16+00:00,0.1200
4,2026-09-03 09:00:05+00:00,0.1200
5,2026-09-03 09:00:32+00:00,0.1200
6,2026-09-03 10:00:16+00:00,0.1200
7,2026-09-03 11:00:17+00:00,0.1200
8,2026-09-03 12:00:26+00:00,0.1200
9,2026-09-03 13:00:17+00:00,0.0600
